In [1]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

### 스마트 창고 출고 지연 예측 — AutoEncoder + Stage1/2 (0416, AE ES+WD)

`0415_preproc_logtarget_s2_predlag_full_weight_sweep_exp`와 **동일한 전처리·CV·트리 앙상블**에, CV **각 fold의 학습 구간만**으로 학습한 **오토인코더** 저차원 표현(`ae_z0` …)을 Stage1/2 입력 피처에 붙인 버전입니다.

- 검증 행: 해당 fold AE로만 인코딩 → **OOF 누수 없음**
- 테스트: fold별 인코더 출력의 **산술 평균**
- 끄기: `USE_AUTOENCODER = False`
- **이 변형(variant)**: fold 검증 MSE **early stopping**(`AE_ES_PATIENCE`, `AE_ES_MIN_DELTA`) + 최저 val 시점 **best state 복원**. Adam `weight_decay=AE_WEIGHT_DECAY`.
- `AE_EPOCHS`: early stopping **최대 에폭(상한)**.

- **MAE boost 패키지**: `AdamW` + `AE_DENOISE_STD` + `AE_DROPOUT`, ES/WD 기본값 재튜닝, `BLEND_ALPHA_STEP=0.005`, 클립 q 그리드 보강, `USE_STAGE1_RESIDUAL`로 S1b 잔차 ON/OFF.


### 실행 파라미터

In [ ]:
SEED = 42
USE_LOG_TARGET = True  # logtarget + s2_predlag one-shot test

# 빠른 실험(기본): N_FOLDS=5, S1_LGB_SEEDS=[SEED]
# 풀 설정: PowerShell에서 `$env:JW_V13_FULL='1'` 후 실행
_V13_FULL = True  # expanded folds/seeds experiment
if _V13_FULL:
    N_FOLDS = 10
else:
    N_FOLDS = 5
S1_LGB_SEEDS = [SEED, SEED + 100, SEED + 200, SEED + 300] if _V13_FULL else [SEED, SEED + 100]

RUN_STRATIFIED_GROUP_CV = True
BLEND_ALPHA_STEP = 0.005  # finer alpha sweep (S1–S2 blend)
REPLACE_STAGE2_IF_NOT_IMPROVING = False  # allow S2 in blend; revisit after OOF/LB
USE_S2_PRED_LAG_ONLY = True  # Stage2 uses Stage1 pred-lag features only
TOP_IMPORTANCE_K = 30

# 요청 반영: 모델에 직접 영향 주는 top10만 사용
USE_TOP10_ONLY = True

# Stage1b residual (S1 앙상블 위 LGB 잔차). OFF면 학습/테스트 모두 pre-ensemble만 사용.
USE_STAGE1_RESIDUAL = False

CLIP_PRED_MIN = 0.0
CLIP_PRED_MAX_Q = 0.995
ENABLE_CLIP_Q_SWEEP = True
CLIP_Q_GRID = [0.992, 0.993, 0.994, 0.995, 0.996, 0.997, 0.998]

ENABLE_SAMPLE_WEIGHT = True
SW_HIGH_Q = 0.90
SW_HIGH_MULT = 1.25
SW_TS_EDGE_MULT = 1.10

PREPROC_ONLY_EXPERIMENT = True
ENABLE_DOMAIN_CLIP = True
ENABLE_MISSING_MASK = True

# ratio feature/column name hints
RATIO_COL_KEYWORDS = ['ratio', 'utilization', 'efficiency', 'pct', 'percent', 'share']

# non-negative metric name hints
NONNEG_COL_KEYWORDS = [
    'count', 'cnt', 'num', 'minutes', 'min', 'hour', 'distance',
    'delay', 'inflow', 'order', 'robot', 'battery', 'density',
    'area', 'weight', 'pressure', 'risk', 'wait', 'trip'
]

# --- (2단계) AE 미세 튜닝: 출발점은 아래 기본값. 한 축만 바꿔 비교 ---
#  1) AE_WEIGHT_DECAY  (예: 1e-5 -> 3e-5 -> 1e-4)
#  2) AE_DROPOUT / AE_DENOISE_STD  (예: dropout 0 vs 0.05, denoise 0 vs 0.01)
#  3) AE_ES_PATIENCE / AE_EPOCHS 상한  (예: patience 6->8, EPOCHS 48->64)

# --- Autoencoder (fold-wise OOF-safe; bottleneck features appended to Stage1/2) ---
USE_AUTOENCODER = True
AE_LATENT_DIM = 32
AE_HIDDEN_DIM = 256
AE_BATCH_SIZE = 4096
AE_LR = 1e-3
# AE_EPOCHS: ES 최대 에폭(상한). val 재구성 MSE + patience + min_delta
AE_EPOCHS = 48
AE_ES_PATIENCE = 8
AE_ES_MIN_DELTA = 1e-6
AE_WEIGHT_DECAY = 3e-5
AE_DENOISE_STD = 0
AE_DROPOUT = 0

# CUDA 사용 가능 시 GPU (대용량 행렬에서 유리)
AE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### 보조 함수 설정

In [3]:
def elapsed(start):
    s = int(time.time() - start)
    return f"{s // 60}m {s % 60:02d}s"


def section(title):
    print(f"\n{'=' * 60}\n  {title}\n{'=' * 60}")


def _grid_01(step: float) -> np.ndarray:
    if step <= 0 or step > 1:
        step = 0.05
    n = max(1, int(round(1.0 / step)))
    return np.linspace(0.0, 1.0, n + 1)


def _resolve_data_dir() -> str:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        d = p / 'data'
        if d.is_dir() and (d / 'train.csv').is_file():
            return str(d)
    raise FileNotFoundError('data/train.csv not found')


def to_train_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y


def from_train_pred(p):
    return np.expm1(p) if USE_LOG_TARGET else p


def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def _build_sample_weight(df: pd.DataFrame) -> np.ndarray:
    """Simple robust weighting: high-target + edge timeslot emphasis."""
    w = np.ones(len(df), dtype=np.float64)
    if not ENABLE_SAMPLE_WEIGHT:
        return w

    # Keep this robust even when cells are run out of order.
    target_col = globals().get('TARGET', 'avg_delay_minutes_next_30m')
    if target_col not in df.columns:
        return w

    y = df[target_col].to_numpy(dtype=np.float64)
    q_thr = float(np.quantile(y, SW_HIGH_Q))
    w[y >= q_thr] *= float(SW_HIGH_MULT)

    if 'timeslot' in df.columns:
        ts = df['timeslot'].to_numpy(dtype=np.int64)
        edge_mask = (ts <= 2) | (ts >= 22)
        w[edge_mask] *= float(SW_TS_EDGE_MULT)
    return w


def _ensemble_pred(oof_by_model: dict[str, np.ndarray], maes_by_model: dict[str, float], models: list[str], p: float) -> np.ndarray:
    w = {m: 1.0 / (maes_by_model[m] ** p) for m in models}
    ws = sum(w.values())
    out = np.zeros_like(next(iter(oof_by_model.values())))
    for m in models:
        out += w[m] * oof_by_model[m]
    return out / ws


def _powerset_models_s1():
    return [
        ['lgb'], ['xgb'], ['cat'],
        ['lgb', 'xgb'], ['lgb', 'cat'], ['xgb', 'cat'],
        ['lgb', 'xgb', 'cat'],
    ]


class TabularAutoEncoder(nn.Module):
    def __init__(self, n_in: int, hidden: int, latent: int, dropout: float = 0.0):
        super().__init__()
        p = float(dropout)
        enc_layers = [nn.Linear(n_in, hidden), nn.ReLU(inplace=True)]
        if p > 0:
            enc_layers.append(nn.Dropout(p))
        enc_layers.append(nn.Linear(hidden, latent))
        self.enc = nn.Sequential(*enc_layers)
        dec_layers = [nn.Linear(latent, hidden), nn.ReLU(inplace=True)]
        if p > 0:
            dec_layers.append(nn.Dropout(p))
        dec_layers.append(nn.Linear(hidden, n_in))
        self.dec = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.enc(x)
        x_hat = self.dec(z)
        return x_hat, z


def _ae_prepare_matrix(df: pd.DataFrame, cols: list[str], medians: pd.Series) -> np.ndarray:
    X = df[cols].to_numpy(dtype=np.float64)
    for j, col in enumerate(cols):
        m = float(medians[col])
        colv = X[:, j]
        colv[~np.isfinite(colv)] = m
        colv[np.isnan(colv)] = m
    return X


def _ae_train_fold(X_tr: np.ndarray, X_va: np.ndarray, device: torch.device, seed: int):
    torch.manual_seed(seed)
    n_in = X_tr.shape[1]
    p_drop = float(globals().get("AE_DROPOUT", 0.0))
    model = TabularAutoEncoder(n_in, AE_HIDDEN_DIM, AE_LATENT_DIM, p_drop).to(device)
    wd = float(globals().get("AE_WEIGHT_DECAY", 0.0))
    opt = torch.optim.AdamW(model.parameters(), lr=AE_LR, weight_decay=wd)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    tr_t = torch.from_numpy(X_tr_s).float().to(device)
    va_t = torch.from_numpy(X_va_s).float().to(device)
    n = tr_t.shape[0]

    max_epochs = int(AE_EPOCHS)
    patience = int(globals().get("AE_ES_PATIENCE", 4))
    min_delta = float(globals().get("AE_ES_MIN_DELTA", 0.0))
    sigma = float(globals().get("AE_DENOISE_STD", 0.0))

    best_val = float("inf")
    best_state = None
    bad_epochs = 0

    for _epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, AE_BATCH_SIZE):
            idx = perm[i : i + AE_BATCH_SIZE]
            xb = tr_t[idx]
            opt.zero_grad(set_to_none=True)
            if sigma > 0:
                x_in = xb + torch.randn_like(xb) * sigma
            else:
                x_in = xb
            xh, _ = model(x_in)
            loss = nn.functional.mse_loss(xh, xb)
            loss.backward()
            opt.step()

        if va_t.shape[0] == 0:
            break

        model.eval()
        with torch.no_grad():
            xh_va, z_va = model(va_t)
            val_loss = nn.functional.mse_loss(xh_va, va_t, reduction="mean").item()

        if val_loss < (best_val - min_delta):
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is None:
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        _, z_va = model(va_t)
    z_va_np = z_va.cpu().numpy()
    state_cpu = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    return scaler, state_cpu, z_va_np


def _ae_encode_matrix(X_scaled: np.ndarray, state_dict: dict, device: torch.device) -> np.ndarray:
    n_in = X_scaled.shape[1]
    p_drop = float(globals().get("AE_DROPOUT", 0.0))
    model = TabularAutoEncoder(n_in, AE_HIDDEN_DIM, AE_LATENT_DIM, p_drop).to(device)
    model.load_state_dict({k: v.to(device) for k, v in state_dict.items()})
    model.eval()
    out = []
    xt = torch.from_numpy(X_scaled).float().to(device)
    with torch.no_grad():
        for i in range(0, len(xt), AE_BATCH_SIZE):
            _, z = model(xt[i : i + AE_BATCH_SIZE])
            out.append(z.cpu().numpy())
    return np.vstack(out)


### 피처 컬럼명 설정

In [4]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

# Full true-target lag column names (used only if USE_S2_PRED_LAG_ONLY == False)
TRUE_LAG_COLS_LEGACY = [
    'target_lag1', 'target_lag2', 'target_lag3', 'target_lag4', 'target_lag5',
    'target_roll3_mean', 'target_roll5_mean', 'target_roll10_mean',
    'target_ewm3', 'target_ewm5',
    'target_diff1', 'target_diff2',
    'target_lag1_log',
    'target_lag_max3', 'target_lag_min3', 'target_lag_std3',
]

TRUE_LAG_COLS = [] if USE_S2_PRED_LAG_ONLY else TRUE_LAG_COLS_LEGACY

PRED_LAG_COLS = [
    'pred_lag1', 'pred_lag2', 'pred_lag3',
    'pred_diff1', 'pred_diff2',
    'pred_roll3_mean', 'pred_roll5_mean',
    'pred_ewm3',
    'pred_lag1_log',
]


### 공동 유틸 : Lag 업데이트 + 순차 예측

In [5]:

def add_pred_lag_features(df, pred_vec, gm):
    """Scenario-wise lags of Stage1 predictions (OOF or test). gm fills cold-start."""
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    p = np.asarray(pred_vec, dtype=np.float64)
    if len(p) != len(df):
        raise ValueError(f'add_pred_lag_features: len(pred_vec)={len(p)} != len(df)={len(df)}')
    df['_pred_for_lag'] = p
    g = df.groupby('scenario_id')['_pred_for_lag']
    df['pred_lag1'] = g.shift(1)
    df['pred_lag2'] = g.shift(2)
    df['pred_lag3'] = g.shift(3)
    s1, s2, s3 = g.shift(1), g.shift(2), g.shift(3)
    df['pred_diff1'] = (s1 - s2).fillna(0.0)
    df['pred_diff2'] = (s2 - s3).fillna(0.0)
    df['pred_roll3_mean'] = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df['pred_roll5_mean'] = g.transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df['pred_ewm3'] = g.transform(lambda x: x.shift(1).ewm(alpha=0.3, adjust=False).mean())
    df['pred_lag1_log'] = np.log1p(np.maximum(df['pred_lag1'].fillna(gm).astype(np.float64), 0.0))
    for c in ['pred_lag1', 'pred_lag2', 'pred_lag3', 'pred_roll3_mean', 'pred_roll5_mean', 'pred_ewm3']:
        df[c] = df[c].fillna(gm)
    df.drop(columns=['_pred_for_lag'], inplace=True)
    return df


def _update_lag_from_history(df, row_indices, scenario_history, gm):
    """
    slot > 0 일 때 row_indices 의 각 행의 TRUE_LAG_COLS를
    해당 시나리오의 예측 히스토리로 갱신한다.
    df는 float64 타입이어야 하며 인덱스가 연속 정수여야 한다.
    """
    for pos in row_indices:
        scen_id = df.at[pos, 'scenario_id']
        hist = scenario_history.get(scen_id, [])
        n = len(hist)

        # 기본 lag
        for lag_i in [1, 2, 3, 4, 5]:
            df.at[pos, f'target_lag{lag_i}'] = hist[-lag_i] if n >= lag_i else gm

        # rolling mean (3, 5, 10)
        for w in [3, 5, 10]:
            df.at[pos, f'target_roll{w}_mean'] = float(np.mean(hist[-w:])) if n > 0 else gm

        # rolling max/min/std (3슬롯)
        df.at[pos, 'target_lag_max3'] = float(np.max(hist[-3:])) if n > 0 else gm
        df.at[pos, 'target_lag_min3'] = float(np.min(hist[-3:])) if n > 0 else gm
        df.at[pos, 'target_lag_std3'] = float(np.std(hist[-3:])) if n >= 2 else 0.0

        # EWM (α=0.3, 0.5)
        if n == 0:
            ewm3, ewm5 = gm, gm
        else:
            ewm3 = ewm5 = hist[0]
            for v in hist[1:]:
                ewm3 = 0.3 * v + 0.7 * ewm3
                ewm5 = 0.5 * v + 0.5 * ewm5
        df.at[pos, 'target_ewm3'] = ewm3
        df.at[pos, 'target_ewm5'] = ewm5

        # 차분
        df.at[pos, 'target_diff1'] = (hist[-1] - hist[-2]) if n >= 2 else 0.0
        df.at[pos, 'target_diff2'] = (hist[-2] - hist[-3]) if n >= 3 else 0.0

        # log lag
        df.at[pos, 'target_lag1_log'] = (
            float(np.log1p(max(hist[-1], 0))) if n >= 1 else float(np.log1p(max(gm, 0)))
        )


def sequential_predict(df_sorted, models_list, feature_cols, gm, from_pred_fn, n_slots=25):
    """
    슬롯별 순차 예측 함수.

    Parameters
    ----------
    df_sorted  : scenario_id, timeslot 순으로 정렬된 DataFrame.
                 TRUE_LAG_COLS 컬럼이 존재해야 하며 gm 으로 초기화된 상태여야 함.
                 인덱스는 호출 측에서 원본 위치를 복원할 수 있도록 보존되어야 한다.
    models_list: fold 모델 리스트 (앙상블 평균).
    feature_cols: 예측에 사용할 컬럼 리스트.
    gm         : global mean (lag 초기값).
    from_pred_fn: 모델 출력 후처리 함수 (log target이면 expm1, 아니면 identity).
    n_slots    : 타임슬롯 수 (default 25).

    Returns
    -------
    preds_out : np.ndarray, df_sorted 행 순서와 동일한 예측값.
    """
    df = df_sorted.copy()
    # lag 피처 초기화 (slot 0 에는 과거 정보 없음)
    for col in TRUE_LAG_COLS:
        df[col] = gm

    preds_out = np.zeros(len(df))
    scenario_history = {}  # {scenario_id: [pred_t0, pred_t1, ...]}

    for slot in range(n_slots):
        slot_mask = df['timeslot'] == slot
        if not slot_mask.any():
            continue
        slot_idx = df.index[slot_mask].tolist()

        # slot > 0: 이전 슬롯 예측값으로 lag 갱신
        if slot > 0:
            _update_lag_from_history(df, slot_idx, scenario_history, gm)

        # 앙상블 평균 예측
        slot_data = df.loc[slot_idx, feature_cols]
        slot_preds = np.mean(
            [from_pred_fn(m.predict(slot_data)) for m in models_list], axis=0
        )
        preds_out[slot_mask.values] = slot_preds

        # 히스토리 누적
        for i, pos in enumerate(slot_idx):
            scen_id = df.at[pos, 'scenario_id']
            scenario_history.setdefault(scen_id, []).append(float(slot_preds[i]))

    return preds_out

### 데이터 로드

In [6]:
path = _resolve_data_dir()
project_root = str(Path(path).resolve().parent)
print(f"▶ data dir: {path}")
print(f"▶ project root: {project_root}")

t0 = time.time()
train = pd.read_csv(os.path.join(path, 'train.csv'))
test = pd.read_csv(os.path.join(path, 'test.csv'))
layout = pd.read_csv(os.path.join(path, 'layout_info.csv'))
print(f"▶ load done ({elapsed(t0)})  train {len(train):,} / test {len(test):,}")
print(
    f"▶ jw_v13: mode={'FULL(JW_V13_FULL=1)' if _V13_FULL else 'quick(default)'}, "
    f"N_FOLDS={N_FOLDS}, S1_LGB seeds={len(S1_LGB_SEEDS)}, blend_step={BLEND_ALPHA_STEP}, true-lag+MethodB"
)

▶ data dir: C:\SM\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition\data
▶ project root: C:\SM\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition
▶ load done (0m 01s)  train 250,000 / test 50,000
▶ jw_v13: mode=FULL(JW_V13_FULL=1), N_FOLDS=10, S1_LGB seeds=4, blend_step=0.005, true-lag+MethodB


## 데이터 전처리

In [7]:
# Preprocessing speed-up experiment:
# - reduce TS_COLS
# - reduce rolling windows
# - apply EWM only on core columns
# - remove repeated sort/reset in helper functions
# - aggregate scenario stats in one groupby/one merge

ROLL_WINDOWS = (3, 10)
TS_COLS = [
    'order_inflow_15m',
    'robot_active',
    'robot_total',
    'pack_utilization',
    'congestion_score',
    'avg_trip_distance',
    'low_battery_ratio',
    'outbound_truck_wait_min',
    'order_per_station',
    'robot_efficiency',
]
EWM_CORE_COLS = ['order_inflow_15m', 'congestion_score', 'pack_utilization']
SCEN_AGG_STATS = ('mean', 'max', 'min', 'std')


def add_missing_indicators(df):
    """Create binary mask columns for missing values (excluding IDs/target)."""
    miss_cols = [c for c in df.columns if c not in ID_COLS + [TARGET] and df[c].isnull().any()]
    for c in miss_cols:
        df[f'{c}_isna'] = df[c].isnull().astype(np.int8)
    return df


def handle_missing_values(df):
    cols = [c for c in df.columns if df[c].isnull().any() and c not in ID_COLS + [TARGET]]
    if cols:
        df[cols] = df.groupby('scenario_id')[cols].ffill()
        df[cols] = df.groupby('scenario_id')[cols].bfill()
        df[cols] = df[cols].fillna(df[cols].median())
    return df


def clip_domain_values(df):
    """Apply simple physical constraints to numeric columns."""
    for c in df.columns:
        if c in ID_COLS + [TARGET]:
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        lc = c.lower()
        if any(k in lc for k in RATIO_COL_KEYWORDS):
            df[c] = df[c].clip(0.0, 1.0)
            continue
        if any(k in lc for k in NONNEG_COL_KEYWORDS):
            df[c] = df[c].clip(lower=0.0)
    return df


def add_basic_features(df):
    df['timeslot'] = df.groupby('scenario_id').cumcount()
    df['robot_efficiency'] = df['robot_active'] / (df['robot_total'] + 1e-6)
    df['robot_density'] = df['robot_total'] / (df['floor_area_sqm'] + 1e-6)
    df['order_per_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
    df['robot_per_station'] = df['robot_active'] / (df['pack_station_count'] + 1e-6)
    df['cumulative_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['order_pressure'] = df['cumulative_orders'] / (df['pack_station_count'] + 1e-6)
    if 'congestion_score' in df.columns:
        df['risk_index'] = df['congestion_score'] * (1 - df['robot_efficiency'])
        df['bottle_neck'] = df['order_per_station'] * df['congestion_score']
    if 'low_battery_ratio' in df.columns:
        df['battery_risk'] = df['low_battery_ratio'] * df['robot_total']
    if 'battery_mean' in df.columns and 'battery_std' in df.columns:
        df['battery_cv'] = df['battery_std'] / (df['battery_mean'] + 1e-6)
    return df


def add_timeseries_features(df):
    scen = df['scenario_id']
    for col in TS_COLS:
        if col not in df.columns:
            continue
        g = df.groupby('scenario_id')[col]

        for lag_n in (1, 2, 3):
            df[f'{col}_lag{lag_n}'] = g.shift(lag_n)

        s1 = g.shift(1)
        s2 = g.shift(2)
        s3 = g.shift(3)
        df[f'{col}_diff1'] = s1 - s2
        df[f'{col}_diff2'] = s2 - s3

        # vectorized group rolling on shifted base series (no transform(lambda ...))
        for w in ROLL_WINDOWS:
            roll = s1.groupby(scen).rolling(window=w, min_periods=1)
            df[f'{col}_roll{w}_mean'] = roll.mean().reset_index(level=0, drop=True)
            df[f'{col}_roll{w}_std'] = roll.std().reset_index(level=0, drop=True).fillna(0.0)

        if col in EWM_CORE_COLS:
            df[f'{col}_ewm_mean'] = s1.groupby(scen).ewm(alpha=0.3, adjust=False).mean().reset_index(level=0, drop=True)

    lag_cols = [c for c in df.columns if ('_lag' in c or '_diff' in c) and c not in ID_COLS]
    if lag_cols:
        df[lag_cols] = df.groupby('scenario_id')[lag_cols].ffill()
        for c in lag_cols:
            base_col = c.split('_lag')[0].split('_diff')[0]
            if base_col in df.columns:
                scen_mean = df.groupby('scenario_id')[base_col].transform('mean')
                df[c] = df[c].fillna(scen_mean)
        df[lag_cols] = df[lag_cols].fillna(df[lag_cols].median())
    return df


def add_interaction_features(df):
    if 'congestion_score' in df.columns and 'pack_utilization' in df.columns:
        df['cong_x_pack'] = df['congestion_score'] * df['pack_utilization']
    if 'congestion_score' in df.columns and 'avg_trip_distance' in df.columns:
        df['cong_x_trip'] = df['congestion_score'] * df['avg_trip_distance']
    if 'low_battery_ratio' in df.columns and 'robot_efficiency' in df.columns:
        df['lowbat_x_eff'] = df['low_battery_ratio'] * (1 - df['robot_efficiency'])
    if 'order_per_station' in df.columns and 'congestion_score' in df.columns:
        df['ops_x_cong'] = df['order_per_station'] * df['congestion_score']
    if 'order_per_station' in df.columns and 'pack_utilization' in df.columns:
        df['ops_x_pack'] = df['order_per_station'] * df['pack_utilization']
    if 'timeslot' in df.columns:
        for col in ['congestion_score', 'pack_utilization', 'order_per_station', 'low_battery_ratio']:
            if col in df.columns:
                df[f'ts_x_{col}'] = df['timeslot'] * df[col]

    scen_agg_cols = [
        'congestion_score', 'order_inflow_15m', 'battery_mean', 'pack_utilization',
        'avg_trip_distance', 'low_battery_ratio', 'max_zone_density', 'sku_concentration',
        'robot_idle', 'outbound_truck_wait_min', 'order_per_station', 'robot_efficiency',
        'order_pressure', 'risk_index', 'battery_risk', 'battery_cv',
    ]
    present = [c for c in scen_agg_cols if c in df.columns]
    if present:
        stats = df.groupby('scenario_id')[present].agg(SCEN_AGG_STATS)
        stats.columns = [f'{c}_scen_{s}' for c, s in stats.columns]
        stats = stats.reset_index()
        df = df.merge(stats, on='scenario_id', how='left')

    for col in ['congestion_score', 'order_per_station', 'pack_utilization', 'avg_trip_distance']:
        sm = f'{col}_scen_mean'
        if col in df.columns and sm in df.columns:
            df[f'{col}_rel_to_scen'] = df[col] / (df[sm] + 1e-6)

    for col in ['congestion_score', 'order_per_station', 'pack_utilization']:
        if col in df.columns:
            df[f'{col}_scen_rank'] = df.groupby('scenario_id')[col].rank(pct=True)
    return df


def add_extra_features_v13(df):
    """Scenario progress + periodic features (train/test shared)."""
    gsz = df.groupby('scenario_id')['scenario_id'].transform('size')
    gpos = df.groupby('scenario_id').cumcount()
    df['v13_row_frac_in_scen'] = (gpos + 1) / (gsz + 1e-6)
    if 'order_inflow_15m' in df.columns:
        df['v13_sqrt_order_inflow'] = np.sqrt(np.maximum(df['order_inflow_15m'].astype(np.float64), 0.0))
    if 'robot_total' in df.columns:
        df['v13_sqrt_robot_total'] = np.sqrt(np.maximum(df['robot_total'].astype(np.float64), 0.0))
    if 'timeslot' in df.columns:
        ang = 2 * np.pi * df['timeslot'].astype(np.float64) / 24.0
        df['v13_ts_sin'] = np.sin(ang)
        df['v13_ts_cos'] = np.cos(ang)
    return df


def preprocess_all(df, layout_df):
    df = df.merge(layout_df, on='layout_id', how='left')
    # Sort once to keep all downstream groupby-shift operations consistent.
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)

    if ENABLE_MISSING_MASK:
        df = add_missing_indicators(df)
    df = handle_missing_values(df)
    df = add_basic_features(df)
    df = add_timeseries_features(df)
    df = add_interaction_features(df)
    df = add_extra_features_v13(df)

    if ENABLE_DOMAIN_CLIP:
        df = clip_domain_values(df)

    if 'layout_type' in df.columns:
        df['layout_type'] = pd.factorize(df['layout_type'])[0]
    return df


section('Preprocess (speedup experiment: reduced TS/rolling + vectorized groupby)')
t0 = time.time()
train = preprocess_all(train, layout)
test = preprocess_all(test, layout)
print(f"▶ preprocess done ({elapsed(t0)})")
if PREPROC_ONLY_EXPERIMENT:
    print('▶ preproc-only experiment mode: model/CV/ensemble logic unchanged')
print(f'▶ TS_COLS={len(TS_COLS)}, ROLL_WINDOWS={ROLL_WINDOWS}, EWM_CORE_COLS={EWM_CORE_COLS}')



  Preprocess (speedup experiment: reduced TS/rolling + vectorized groupby)
▶ preprocess done (0m 13s)
▶ preproc-only experiment mode: model/CV/ensemble logic unchanged
▶ TS_COLS=10, ROLL_WINDOWS=(3, 10), EWM_CORE_COLS=['order_inflow_15m', 'congestion_score', 'pack_utilization']


### 타겟 인코딩

In [8]:
section('Target Encoding')
t0 = time.time()
TE_COLS = [c for c in ['layout_id', 'timeslot', 'layout_type', 'shift_hour', 'day_of_week'] if c in train.columns]
TE_PAIRS = []
for a in TE_COLS:
    for b in TE_COLS:
        if a < b:
            TE_PAIRS.append((a, b))

SMOOTHING = 20
kf_te = GroupKFold(n_splits=N_FOLDS)
groups_te = train['scenario_id']
global_mean = float(train[TARGET].mean())


def _apply_te(df_train, df_test, col_name, group_col_series_tr, group_col_series_te):
    te_col = f'{col_name}_te'
    df_train[te_col] = np.nan
    for tr_idx, val_idx in kf_te.split(df_train, df_train[TARGET], groups=groups_te):
        tr_df = df_train.iloc[tr_idx]
        stats = tr_df.groupby(group_col_series_tr.iloc[tr_idx])[TARGET].agg(['mean', 'count'])
        smooth = (stats['count'] * stats['mean'] + SMOOTHING * global_mean) / (stats['count'] + SMOOTHING)
        df_train.loc[df_train.index[val_idx], te_col] = group_col_series_tr.iloc[val_idx].map(smooth).fillna(global_mean)
    stats_full = df_train.groupby(group_col_series_tr)[TARGET].agg(['mean', 'count'])
    smooth_full = (stats_full['count'] * stats_full['mean'] + SMOOTHING * global_mean) / (stats_full['count'] + SMOOTHING)
    df_test[te_col] = group_col_series_te.map(smooth_full).fillna(global_mean)


for col in TE_COLS:
    _apply_te(train, test, col, train[col], test[col])

for a, b in TE_PAIRS:
    pair_name = f'{a}_X_{b}'
    tr_key = train[a].astype(str) + '_' + train[b].astype(str)
    te_key = test[a].astype(str) + '_' + test[b].astype(str)
    _apply_te(train, test, pair_name, tr_key, te_key)

print(f"▶ target encoding done ({elapsed(t0)})")


  Target Encoding
▶ target encoding done (0m 52s)


### 실제 Target Lag feature 생성 (train only)

In [9]:
section('True Target Lag Feature Engineering (train only)')
t0 = time.time()
if USE_S2_PRED_LAG_ONLY:
    print('▶ Skipped true-target lag features (Stage2 uses Stage1 pred-lag columns only)')
else:
    train = train.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    g_target = train.groupby('scenario_id')[TARGET]

    for lag in [1, 2, 3, 4, 5]:
        train[f'target_lag{lag}'] = g_target.shift(lag).fillna(global_mean)

    for w in [3, 5, 10]:
        train[f'target_roll{w}_mean'] = g_target.transform(
            lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean()
        ).fillna(global_mean)

    train['target_lag_max3'] = g_target.transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).max()
    ).fillna(global_mean)
    train['target_lag_min3'] = g_target.transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).min()
    ).fillna(global_mean)
    train['target_lag_std3'] = g_target.transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0)
    ).fillna(0)

    train['target_ewm3'] = g_target.transform(
        lambda x: x.shift(1).ewm(alpha=0.3, adjust=False).mean()
    ).fillna(global_mean)
    train['target_ewm5'] = g_target.transform(
        lambda x: x.shift(1).ewm(alpha=0.5, adjust=False).mean()
    ).fillna(global_mean)

    train['target_diff1'] = train['target_lag1'] - train['target_lag2']
    train['target_diff2'] = train['target_lag2'] - train['target_lag3']
    train['target_lag1_log'] = np.log1p(train['target_lag1'].clip(lower=0))

    for col in TRUE_LAG_COLS_LEGACY:
        if col in train.columns:
            train[col] = train[col].fillna(global_mean)

    print(f"  target_lag1 corr with target: {train['target_lag1'].corr(train[TARGET]):.4f}")
    print(f"  target_lag2 corr with target: {train['target_lag2'].corr(train[TARGET]):.4f}")
    print(f"  target_ewm3 corr with target: {train['target_ewm3'].corr(train[TARGET]):.4f}")
    print(f"  TRUE_LAG_COLS_LEGACY count: {len(TRUE_LAG_COLS_LEGACY)}")

print(f"▶ True target lag block done ({elapsed(t0)})")



  True Target Lag Feature Engineering (train only)
▶ Skipped true-target lag features (Stage2 uses Stage1 pred-lag columns only)
▶ True target lag block done (0m 00s)


# Stage 1
- target lag 없음
- 실제 target lag feature에서 train에 lag 컬럼이 추가된 상태이므로 제외하지 않으면 Stage 1이 real lag로 학습
- TRUE_LAG_COLS를 명시적으로 제외

### feature_cols_s1
- ID 제외
- target 제외
- 실제 target lag 컬럼 제외
    - 이미 train에 실제 target lag 컬럼이 만들어져 있는데, stage 1에 넣어버리면 검증 점수가 비정상적으로 좋아질 수 있으므로, 실제 test 상황과 달라져 OOF와 리더보드 점수 차이가 커질 수 있음

### CV 방식
- 가능하면 StratifiedGroupKFold, 안 되면 GroupKFold

## 모델
- LightGBM
- XGBoost
- CatBoost

- 폴드마다 3개 모델을 학습하고 OOF 예측을 만듦

## Stage 1 앙상블
- 각 모델의 OOF MAE를 보고, 모델 조합과 p값을 바꿔가며 최적 가중 평균을 찾음

### Stage 1b Residual
- Stage 1 앙상블 예측의 오차를 다시 LightGBM으로 학습

In [10]:
feature_cols_s1_base = [
    c for c in train.columns
    if c not in ID_COLS + [TARGET] + TRUE_LAG_COLS
]
ae_input_cols = [
    c for c in feature_cols_s1_base
    if pd.api.types.is_numeric_dtype(train[c])
]
AE_COLS = [f"ae_z{i}" for i in range(AE_LATENT_DIM)] if USE_AUTOENCODER else []
ae_fold_artifacts = []

if USE_AUTOENCODER:
    for cname in AE_COLS:
        train[cname] = 0.0
        test[cname] = 0.0
    feature_cols_s1 = feature_cols_s1_base + AE_COLS
    print(
        f"▶ Stage 1 features: {len(feature_cols_s1)}  "
        f"(base={len(feature_cols_s1_base)} + AE={len(AE_COLS)}, device={AE_DEVICE})"
    )
else:
    feature_cols_s1 = list(feature_cols_s1_base)
    ae_input_cols = []
    print(f"▶ Stage 1 features: {len(feature_cols_s1)}  (TRUE_LAG_COLS excluded, AE off)")

y_all = to_train_target(train[TARGET].values)
y_raw = train[TARGET].values
groups = train['scenario_id'].values
sw_all = _build_sample_weight(train)
print(f"▶ sample_weight enabled={ENABLE_SAMPLE_WEIGHT} | mean={sw_all.mean():.4f} | max={sw_all.max():.3f}")

_scen_mean = train.groupby('scenario_id')[TARGET].transform('mean')
try:
    y_strat_cv = pd.qcut(_scen_mean, q=10, labels=False, duplicates='drop')
    y_strat_cv = pd.Series(y_strat_cv).fillna(0).astype(np.int64).values
except Exception:
    y_strat_cv = np.zeros(len(train), dtype=np.int64)

kf_y = y_strat_cv
if RUN_STRATIFIED_GROUP_CV:
    try:
        from sklearn.model_selection import StratifiedGroupKFold

        kf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        kf_y = y_strat_cv
        print('▶ CV: StratifiedGroupKFold (scenario target deciles)')
    except Exception as e:
        kf = GroupKFold(n_splits=N_FOLDS)
        kf_y = y_all
        print(f'▶ CV: GroupKFold (StratifiedGroupKFold unavailable: {e})')
else:
    kf = GroupKFold(n_splits=N_FOLDS)
    kf_y = y_all
    print('▶ CV: GroupKFold')

lgb_params_s1 = dict(
    objective='regression_l1',
    n_estimators=25000,
    learning_rate=0.01,
    max_depth=-1,
    num_leaves=2047,
    min_child_samples=60,
    subsample=0.75,
    subsample_freq=1,
    colsample_bytree=0.5,
    reg_alpha=0.3,
    reg_lambda=5.0,
    random_state=SEED,
    verbose=-1,
)

xgb_params_s1 = dict(
    objective='reg:absoluteerror',
    n_estimators=20000,
    learning_rate=0.015,
    max_depth=10,
    subsample=0.75,
    colsample_bytree=0.5,
    colsample_bynode=0.5,
    reg_alpha=0.3,
    reg_lambda=3.0,
    random_state=SEED,
    tree_method='hist',
    eval_metric='mae',
    early_stopping_rounds=500,
    verbosity=0,
)

cat_params_s1 = dict(
    iterations=20000,
    learning_rate=0.015,
    depth=10,
    l2_leaf_reg=5.0,
    bootstrap_type='MVS',
    subsample=0.75,
    colsample_bylevel=0.5,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=SEED,
    task_type='CPU',
    early_stopping_rounds=500,
)

section('Stage 1 - Base model (LGB + XGB + Cat + ensemble)')
if len(S1_LGB_SEEDS) > 1:
    print(f"▶ S1 LGB multi-seed (fold 내 평균): {S1_LGB_SEEDS}")
t0 = time.time()
oof_s1_lgb = np.zeros(len(train))
oof_s1_xgb = np.zeros(len(train))
oof_s1_cat = np.zeros(len(train))
models_s1_lgb, models_s1_xgb, models_s1_cat = [], [], []

for fold, (tr_idx, va_idx) in enumerate(kf.split(train, kf_y, groups=groups), 1):
    if USE_AUTOENCODER:
        med_ae = train.iloc[tr_idx][ae_input_cols].median()
        X_ae_tr = _ae_prepare_matrix(train.iloc[tr_idx], ae_input_cols, med_ae)
        X_ae_va = _ae_prepare_matrix(train.iloc[va_idx], ae_input_cols, med_ae)
        sc_ae, sd_ae, z_va = _ae_train_fold(X_ae_tr, X_ae_va, AE_DEVICE, SEED + fold)
        train.loc[train.index[va_idx], AE_COLS] = z_va
        ae_fold_artifacts.append((sc_ae, sd_ae))
        print(f"  [AE] fold {fold}: val embedding shape {z_va.shape}")

    X_tr = train.iloc[tr_idx][feature_cols_s1]
    X_va = train.iloc[va_idx][feature_cols_s1]
    y_tr = y_all[tr_idx]
    y_va = y_all[va_idx]
    sw_tr = sw_all[tr_idx]

    lgb_va_stack = []
    for rs in S1_LGB_SEEDS:
        lp = {**lgb_params_s1, 'random_state': rs}
        m_lgb = lgb.LGBMRegressor(**lp)
        m_lgb.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='mae',
            callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)],
        )
        lgb_va_stack.append(from_train_pred(m_lgb.predict(X_va)))
        models_s1_lgb.append(m_lgb)
    oof_s1_lgb[va_idx] = np.mean(lgb_va_stack, axis=0)

    try:
        m_xgb = xgb.XGBRegressor(**xgb_params_s1)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception:
        xgb_fb = dict(xgb_params_s1)
        xgb_fb['objective'] = 'reg:squarederror'
        m_xgb = xgb.XGBRegressor(**xgb_fb)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_s1_xgb[va_idx] = from_train_pred(m_xgb.predict(X_va))
    models_s1_xgb.append(m_xgb)

    m_cat = cb.CatBoostRegressor(**cat_params_s1)
    m_cat.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=(X_va, y_va),
        verbose=max(1, cat_params_s1['iterations'] // 10),
        use_best_model=True,
    )
    oof_s1_cat[va_idx] = from_train_pred(m_cat.predict(X_va))
    models_s1_cat.append(m_cat)

    ens = (oof_s1_lgb[va_idx] + oof_s1_xgb[va_idx] + oof_s1_cat[va_idx]) / 3
    print(f"  S1 Fold {fold} MAE (avg3): {mae(y_raw[va_idx], ens):.6f}")

mae_s1_lgb = mae(y_raw, oof_s1_lgb)
mae_s1_xgb = mae(y_raw, oof_s1_xgb)
mae_s1_cat = mae(y_raw, oof_s1_cat)
print(f"\n▶ Stage 1 OOF MAE — LGB {mae_s1_lgb:.6f} | XGB {mae_s1_xgb:.6f} | CAT {mae_s1_cat:.6f}")

s1_maes = {'lgb': mae_s1_lgb, 'xgb': mae_s1_xgb, 'cat': mae_s1_cat}
s1_oof_by = {'lgb': oof_s1_lgb, 'xgb': oof_s1_xgb, 'cat': oof_s1_cat}

best_s1_models = ['lgb', 'xgb', 'cat']
best_s1_p = 2.0
best_s1_mae = float('inf')
for models in _powerset_models_s1():
    for p in [1.0, 2.0, 3.0, 4.0]:
        pred_tmp = _ensemble_pred(s1_oof_by, s1_maes, models, p)
        m_val = mae(y_raw, pred_tmp)
        if m_val < best_s1_mae:
            best_s1_mae = float(m_val)
            best_s1_models = list(models)
            best_s1_p = float(p)

print(f"▶ Best Stage1 ensemble: models={best_s1_models}  p={best_s1_p}  OOF_MAE={best_s1_mae:.6f}")

w_s1 = {m: 1.0 / (s1_maes[m] ** best_s1_p) for m in best_s1_models}
ws_s1 = sum(w_s1.values())
oof_s1_pre = sum(w_s1[m] * s1_oof_by[m] for m in best_s1_models) / ws_s1
s1_pre_mae = mae(y_raw, oof_s1_pre)
print(f"▶ Stage 1 ensemble OOF MAE (pre-residual): {s1_pre_mae:.6f}  ({elapsed(t0)})")

# Stage 1b: residual (OOF-safe) — base 예측 오차를 추가로 학습 (플래그로 ON/OFF)
if USE_STAGE1_RESIDUAL:
    section('Stage 1b - Residual LGB (stack on Stage1 ensemble)')
    resid_params = dict(
        objective='regression_l1',
        n_estimators=8000,
        learning_rate=0.02,
        num_leaves=127,
        min_child_samples=40,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.2,
        reg_lambda=2.0,
        random_state=SEED,
        verbose=-1,
    )
    oof_s1_resid = np.zeros(len(train))
    models_s1_resid = []
    t1b = time.time()
    for fold, (tr_idx, va_idx) in enumerate(kf.split(train, kf_y, groups=groups), 1):
        X_tr = train.iloc[tr_idx][feature_cols_s1]
        X_va = train.iloc[va_idx][feature_cols_s1]
        r_tr = y_raw[tr_idx] - oof_s1_pre[tr_idx]
        r_va = y_raw[va_idx] - oof_s1_pre[va_idx]
        sw_tr = sw_all[tr_idx]
        mr = lgb.LGBMRegressor(**resid_params)
        mr.fit(
            X_tr, r_tr,
            sample_weight=sw_tr,
            eval_set=[(X_va, r_va)],
            eval_metric='mae',
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(-1)],
        )
        oof_s1_resid[va_idx] = mr.predict(X_va)
        models_s1_resid.append(mr)
        print(f"  S1b Fold {fold} residual MAE: {mae(r_va, oof_s1_resid[va_idx]):.6f}")

    oof_s1 = oof_s1_pre + oof_s1_resid
    s1_mae = mae(y_raw, oof_s1)
    print(f"\n▶ Stage 1 after residual OOF MAE: {s1_mae:.6f}  ({elapsed(t1b)})")
else:
    oof_s1_resid = np.zeros(len(train))
    models_s1_resid = []
    oof_s1 = np.asarray(oof_s1_pre, dtype=np.float64).copy()
    s1_mae = float(s1_pre_mae)
    print("\n▶ Stage 1 residual: OFF  (oof_s1 = pre-ensemble only)")
if USE_AUTOENCODER and ae_fold_artifacts:
    med_te = train[ae_input_cols].median()
    X_te_raw = _ae_prepare_matrix(test, ae_input_cols, med_te)
    z_folds = []
    for sc_ae, sd_ae in ae_fold_artifacts:
        X_te_s = sc_ae.transform(X_te_raw)
        z_folds.append(_ae_encode_matrix(X_te_s, sd_ae, AE_DEVICE))
    z_mean = np.mean(z_folds, axis=0)
    for i, cname in enumerate(AE_COLS):
        test[cname] = z_mean[:, i]
    print(f"▶ AE test embeddings: mean over {len(ae_fold_artifacts)} fold models")

X_test_s1 = test[feature_cols_s1]
p_lgb = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_lgb], axis=0)
p_xgb = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_xgb], axis=0)
p_cat = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_cat], axis=0)
pred_s1_pre = sum(w_s1[m] * {'lgb': p_lgb, 'xgb': p_xgb, 'cat': p_cat}[m] for m in best_s1_models) / ws_s1
if USE_STAGE1_RESIDUAL and models_s1_resid:
    pred_resid_test = np.mean([m.predict(X_test_s1) for m in models_s1_resid], axis=0)
    pred_s1_test = pred_s1_pre + pred_resid_test
    print(f"▶ Stage 1 test predictions ready (ensemble + residual)")
else:
    pred_s1_test = pred_s1_pre
    print(f"▶ Stage 1 test predictions ready (ensemble only, residual OFF)")

# --- Stage2 pred-lag features (from Stage1 OOF; scenario-safe under group CV) ---
if USE_S2_PRED_LAG_ONLY:
    train = add_pred_lag_features(train, oof_s1, global_mean)
    print(f"▶ Added train pred-lag cols: {PRED_LAG_COLS}  (n={len(PRED_LAG_COLS)})")


▶ Stage 1 features: 426  (base=394 + AE=32, device=cpu)
▶ sample_weight enabled=True | mean=1.0495 | max=1.375
▶ CV: StratifiedGroupKFold (scenario target deciles)

  Stage 1 - Base model (LGB + XGB + Cat + ensemble)
▶ S1 LGB multi-seed (fold 내 평균): [42, 142, 242, 342]
  [AE] fold 1: val embedding shape (25000, 32)
0:	learn: 0.8948700	test: 0.8697581	best: 0.8697581 (0)	total: 354ms	remaining: 1h 57m 56s
2000:	learn: 0.2767794	test: 0.4305237	best: 0.4302028 (1767)	total: 7m 13s	remaining: 1h 4m 56s
Stopped by overfitting detector  (500 iterations wait)

bestTest = 0.4302027768
bestIteration = 1767

Shrink model to first 1768 iterations.
  S1 Fold 1 MAE (avg3): 8.360263
  [AE] fold 2: val embedding shape (25000, 32)
0:	learn: 0.8948010	test: 0.8680368	best: 0.8680368 (0)	total: 224ms	remaining: 1h 14m 34s
2000:	learn: 0.2760389	test: 0.4382058	best: 0.4381879 (1999)	total: 7m 55s	remaining: 1h 11m 15s
Stopped by overfitting detector  (500 iterations wait)

bestTest = 0.4374251325
bestI

# Stage 2
- Stage 2는 Stage 1보다 강한 시계열 모델
- `feature_cols_s2 = feature_cols_s1 + TRUE_LAG_COLS` 로, 실제 target lag 피처까지 포함

## 학습
- 각 fold에서 training fold : 실제 target lag 사용 / validation fold : early stopping 용으로도 실제 lag 사용

## OOF 평가 방식
- 검증 예측은 그냥 predict(X_val) 하지 않고, 반드시 sequential_predict()를 사용해 slot 단위 순차 예측을 수행

## 앙상블
- Stage 2 내부 모델들(LGB, XGB, CAT)을 어떻게 섞을지 탐색

### 결과

In [11]:
feature_cols_s2 = feature_cols_s1 + (PRED_LAG_COLS if USE_S2_PRED_LAG_ONLY else TRUE_LAG_COLS)

lgb_params_s2 = dict(
    objective='regression_l1',
    n_estimators=12000,
    learning_rate=0.02,
    max_depth=-1,
    num_leaves=511,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.65,
    reg_alpha=0.15,
    reg_lambda=4.0,
    random_state=SEED,
    verbose=-1,
)

xgb_params_s2 = dict(
    objective='reg:absoluteerror',
    n_estimators=8000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.65,
    colsample_bynode=0.5,
    reg_alpha=0.15,
    reg_lambda=2.5,
    random_state=SEED,
    tree_method='hist',
    eval_metric='mae',
    early_stopping_rounds=120,
    verbosity=0,
)

cat_params_s2 = dict(
    iterations=8000,
    learning_rate=0.02,
    depth=6,
    l2_leaf_reg=4.0,
    bootstrap_type='MVS',
    subsample=0.8,
    colsample_bylevel=0.65,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=SEED,
    task_type='CPU',
    early_stopping_rounds=120,
)

section('Stage 2 - Pred-lag stack (LGB + XGB + CAT), fold OOF')
t0 = time.time()

if USE_S2_PRED_LAG_ONLY and all(c in train.columns for c in PRED_LAG_COLS):
    pl = train['pred_lag1'].values
    print(f"  pred_lag1 stats: mean={np.nanmean(pl):.2f}  std={np.nanstd(pl):.2f}")
    print(f"  pred_lag1 corr with target: {np.corrcoef(np.nan_to_num(pl), y_raw)[0,1]:.4f}")
else:
    print(f"  target_lag1 stats: mean={train['target_lag1'].mean():.2f}  std={train['target_lag1'].std():.2f}")
    print(f"  target_lag1 corr with target: {np.corrcoef(train['target_lag1'], y_raw)[0,1]:.4f}")

if USE_S2_PRED_LAG_ONLY:
    print('  [Pred-lag S2] OOF = direct predict on each fold val (lags from Stage1 OOF).')
else:
    print('  [Method B] OOF는 slotwise sequential 예측으로 평가 → OOF-LB 갭 제거')

oof_s2_lgb = np.zeros(len(train))
oof_s2_xgb = np.zeros(len(train))
oof_s2_cat = np.zeros(len(train))
models_s2_lgb, models_s2_xgb, models_s2_cat = [], [], []

for fold, (tr_idx, va_idx) in enumerate(kf.split(train, kf_y, groups=groups), 1):
    print(f"\n  --- Stage 2 Fold {fold} ---")

    X_tr = train.iloc[tr_idx][feature_cols_s2]
    y_tr = y_all[tr_idx]

    X_va_real = train.iloc[va_idx][feature_cols_s2]
    y_va_real = y_all[va_idx]
    sw_tr = sw_all[tr_idx]

    m_lgb = lgb.LGBMRegressor(**lgb_params_s2)
    m_lgb.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=[(X_va_real, y_va_real)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(120, verbose=False), lgb.log_evaluation(-1)],
    )
    models_s2_lgb.append(m_lgb)

    try:
        m_xgb = xgb.XGBRegressor(**xgb_params_s2)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va_real, y_va_real)], verbose=False)
    except Exception:
        xgb_fb = dict(xgb_params_s2)
        xgb_fb['objective'] = 'reg:squarederror'
        m_xgb = xgb.XGBRegressor(**xgb_fb)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va_real, y_va_real)], verbose=False)
    models_s2_xgb.append(m_xgb)

    m_cat = cb.CatBoostRegressor(**cat_params_s2)
    m_cat.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=(X_va_real, y_va_real),
        verbose=False,
        use_best_model=True,
    )
    models_s2_cat.append(m_cat)

    if USE_S2_PRED_LAG_ONLY:
        X_va = train.iloc[va_idx][feature_cols_s2]
        oof_s2_lgb[va_idx] = from_train_pred(m_lgb.predict(X_va))
        oof_s2_xgb[va_idx] = from_train_pred(m_xgb.predict(X_va))
        oof_s2_cat[va_idx] = from_train_pred(m_cat.predict(X_va))
    else:
        va_df = train.iloc[va_idx].sort_values(['scenario_id', 'timeslot'])
        lgb_seq = sequential_predict(va_df, [m_lgb], feature_cols_s2, global_mean, from_train_pred)
        for orig_pos, pred in zip(va_df.index, lgb_seq):
            oof_s2_lgb[orig_pos] = pred
        xgb_seq = sequential_predict(va_df, [m_xgb], feature_cols_s2, global_mean, from_train_pred)
        for orig_pos, pred in zip(va_df.index, xgb_seq):
            oof_s2_xgb[orig_pos] = pred
        cat_seq = sequential_predict(va_df, [m_cat], feature_cols_s2, global_mean, from_train_pred)
        for orig_pos, pred in zip(va_df.index, cat_seq):
            oof_s2_cat[orig_pos] = pred

    avg3 = (oof_s2_lgb[va_idx] + oof_s2_xgb[va_idx] + oof_s2_cat[va_idx]) / 3
    tag = 'fold val' if USE_S2_PRED_LAG_ONLY else 'sequential'
    print(f"    LGB MAE ({tag}): {mae(y_raw[va_idx], oof_s2_lgb[va_idx]):.6f}")
    print(f"    XGB MAE ({tag}): {mae(y_raw[va_idx], oof_s2_xgb[va_idx]):.6f}")
    print(f"    CAT MAE ({tag}): {mae(y_raw[va_idx], oof_s2_cat[va_idx]):.6f}")
    print(f"    AVG MAE ({tag}): {mae(y_raw[va_idx], avg3):.6f}")

mae_s2_lgb = mae(y_raw, oof_s2_lgb)
mae_s2_xgb = mae(y_raw, oof_s2_xgb)
mae_s2_cat = mae(y_raw, oof_s2_cat)
_oot = 'Pred-lag fold OOF' if USE_S2_PRED_LAG_ONLY else 'Method B'
print(f"\n▶ Stage 2 OOF MAE ({_oot}) - LGB {mae_s2_lgb:.6f} | XGB {mae_s2_xgb:.6f} | CAT {mae_s2_cat:.6f}")


section('Stage 2 Ensemble search')
s2_model_maes = {'lgb': mae_s2_lgb, 'xgb': mae_s2_xgb, 'cat': mae_s2_cat}
s2_oof_by = {'lgb': oof_s2_lgb, 'xgb': oof_s2_xgb, 'cat': oof_s2_cat}

def _powerset():
    return [
        ["lgb"], ["xgb"], ["cat"],
        ["lgb", "xgb"], ["lgb", "cat"], ["xgb", "cat"],
        ["lgb", "xgb", "cat"],
    ]

best_s2_models = None
best_s2_p = None
best_s2_mae = float('inf')
for models in _powerset():
    for p in [1.0, 2.0, 3.0, 4.0]:
        w = {m: 1.0 / (s2_model_maes[m] ** p) for m in models}
        ws = sum(w.values())
        pred_tmp = sum(w[m] * s2_oof_by[m] for m in models) / ws
        m_val = mae(y_raw, pred_tmp)
        if m_val < best_s2_mae:
            best_s2_mae = float(m_val)
            best_s2_models = list(models)
            best_s2_p = float(p)

print(f"▶ Best S2 ensemble: models={best_s2_models}  p={best_s2_p}  OOF_MAE={best_s2_mae:.6f}")

w_s2 = {m: 1.0 / (s2_model_maes[m] ** best_s2_p) for m in best_s2_models}
ws_s2 = sum(w_s2.values())
oof_s2_ens = sum(w_s2[m] * s2_oof_by[m] for m in best_s2_models) / ws_s2
s2_ens_mae = mae(y_raw, oof_s2_ens)
print(f"▶ Stage 2 Ensemble OOF MAE: {s2_ens_mae:.6f}")

print(f"\n▶ Stage 1 OOF MAE: {s1_mae:.6f}")
print(f"▶ Stage 2 OOF MAE: {s2_ens_mae:.6f}")
print(f"▶ Improvement: {s1_mae - s2_ens_mae:.6f}")
oof_s2_used = oof_s2_ens
use_s1_instead_of_s2 = False
if REPLACE_STAGE2_IF_NOT_IMPROVING and s2_ens_mae >= s1_mae:
    print('▶ REPLACE: Stage2 OOF >= Stage1 → 블렌드/제출은 S2 대신 S1 OOF 사용')
    oof_s2_used = oof_s1.copy()
    use_s1_instead_of_s2 = True



  Stage 2 - Pred-lag stack (LGB + XGB + CAT), fold OOF
  pred_lag1 stats: mean=15.67  std=13.64
  pred_lag1 corr with target: 0.6231
  [Pred-lag S2] OOF = direct predict on each fold val (lags from Stage1 OOF).

  --- Stage 2 Fold 1 ---
    LGB MAE (fold val): 8.437642
    XGB MAE (fold val): 8.417195
    CAT MAE (fold val): 8.407730
    AVG MAE (fold val): 8.397261

  --- Stage 2 Fold 2 ---
    LGB MAE (fold val): 8.338468
    XGB MAE (fold val): 8.325059
    CAT MAE (fold val): 8.398338
    AVG MAE (fold val): 8.327689

  --- Stage 2 Fold 3 ---
    LGB MAE (fold val): 8.208051
    XGB MAE (fold val): 8.146149
    CAT MAE (fold val): 8.174260
    AVG MAE (fold val): 8.148558

  --- Stage 2 Fold 4 ---
    LGB MAE (fold val): 8.475790
    XGB MAE (fold val): 8.462500
    CAT MAE (fold val): 8.480827
    AVG MAE (fold val): 8.442055

  --- Stage 2 Fold 5 ---
    LGB MAE (fold val): 8.691466
    XGB MAE (fold val): 8.719477
    CAT MAE (fold val): 8.766060
    AVG MAE (fold val): 8.69165

## 블렌딩
- `alpha = 1`이면 Stage 1만
- `alpha = 0`이면 Stage 2만
- 그 사이 값이면 둘의 혼합

In [12]:
section('Stage 1 + Stage 2 blending')
best_alpha = 0.0
best_blend_mae = mae(y_raw, oof_s2_used)
best_blend_oof = oof_s2_used.copy()
_alpha_grid = _grid_01(BLEND_ALPHA_STEP)
for i_alpha, alpha in enumerate(_alpha_grid):
    blend_pred = alpha * oof_s1 + (1 - alpha) * oof_s2_used
    m_val = mae(y_raw, blend_pred)
    if m_val < best_blend_mae:
        best_blend_mae = float(m_val)
        best_alpha = float(alpha)
        best_blend_oof = blend_pred.copy()
    if i_alpha in (0, len(_alpha_grid) // 2, len(_alpha_grid) - 1):
        print(f"  alpha={alpha:.4f} MAE={m_val:.6f}")

print(f"\n▶ Best blend: alpha={best_alpha:.2f} (S1 weight)  MAE={best_blend_mae:.6f}")

best_clip_q = float(CLIP_PRED_MAX_Q)
best_clip_mae = best_blend_mae
if ENABLE_CLIP_Q_SWEEP:
    for q in CLIP_Q_GRID:
        q = float(q)
        hi = float(np.percentile(y_raw, 100 * q))
        clipped = np.minimum(np.maximum(best_blend_oof, CLIP_PRED_MIN), hi)
        m_val = mae(y_raw, clipped)
        print(f"  clip_q={q:.4f} MAE={m_val:.6f}")
        if m_val < best_clip_mae:
            best_clip_mae = float(m_val)
            best_clip_q = q
else:
    hi = float(np.percentile(y_raw, 100 * best_clip_q))
    best_clip_mae = mae(y_raw, np.minimum(np.maximum(best_blend_oof, CLIP_PRED_MIN), hi))
print(f"▶ Best clip q={best_clip_q:.4f} MAE={best_clip_mae:.6f}")

# ============================================================
# 8-B) Expert Head 제거
#   이유: slot 1+ OOF 평가에 real target lag 사용 → 동일한 OOF-LB 갭 문제 존재
#        Method B를 적용하면 Stage 2 ensemble이 이미 honest OOF를 반영하므로
#        expert head 없이 Stage1+Stage2 blend가 최종 출력
# ============================================================
best_final_name = "stage1_stage2_blend"
best_final_mae = best_clip_mae
print(f"\n▶ Best final: {best_final_name}  MAE={best_final_mae:.6f}")
print(f"\n▶▶ FINAL OOF MAE: {best_final_mae:.6f}")


  Stage 1 + Stage 2 blending
  alpha=0.0000 MAE=8.464552
  alpha=0.5000 MAE=8.452653
  alpha=1.0000 MAE=8.487858

▶ Best blend: alpha=0.39 (S1 weight)  MAE=8.451368
  clip_q=0.9920 MAE=8.451366
  clip_q=0.9930 MAE=8.451366
  clip_q=0.9940 MAE=8.451366
  clip_q=0.9950 MAE=8.451366
  clip_q=0.9960 MAE=8.451366
  clip_q=0.9970 MAE=8.451366
  clip_q=0.9980 MAE=8.451366
▶ Best clip q=0.9920 MAE=8.451366

▶ Best final: stage1_stage2_blend  MAE=8.451366

▶▶ FINAL OOF MAE: 8.451366


## Test 예측 + 제출

### test 정렬 및 lag 초기화

### Stage 1 test 예측
- Stage 1은 lag 없이 예측 가능하므로 바로 예측

- LGB / XGB / CAT 예측
- Stage 1 가중 앙상블
- residual 보정
- 최종 `pred_s1_test`

### Stage 2 test 예측
- `sequential_predict()`를 사용

- slot 0 예측
- 예측값을 history에 저장
- slot 1의 lag 갱신
- slot 1 예측
- 반복

⇒ 각 모델의 순차 예측 결과를 다시 가중 앙상블해서 `pred_s2_test`를 만듦

### 최종 블렌드 및 후처리

### 제출 파일 저장

In [13]:
section('Predict test + submit (pred-lag S2: direct predict on test)')

test = test.sort_values(['scenario_id', 'timeslot', 'ID']).reset_index(drop=True)
if not USE_S2_PRED_LAG_ONLY:
    for col in TRUE_LAG_COLS:
        test[col] = global_mean

print("  Stage 1 test prediction (no lag)...")
X_test_s1 = test[feature_cols_s1]
p_lgb_s1 = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_lgb], axis=0)
p_xgb_s1 = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_xgb], axis=0)
p_cat_s1 = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_cat], axis=0)
pred_s1_pre = sum(w_s1[m] * {'lgb': p_lgb_s1, 'xgb': p_xgb_s1, 'cat': p_cat_s1}[m]
                  for m in best_s1_models) / ws_s1
if USE_STAGE1_RESIDUAL and models_s1_resid:
    pred_resid_test = np.mean([m.predict(X_test_s1) for m in models_s1_resid], axis=0)
    pred_s1_test = pred_s1_pre + pred_resid_test
else:
    pred_s1_test = pred_s1_pre
print(f"  Stage 1 test pred: mean={pred_s1_test.mean():.2f}  std={pred_s1_test.std():.2f}")

if USE_S2_PRED_LAG_ONLY:
    test = add_pred_lag_features(test, pred_s1_test, global_mean)
    print(f"  Stage 2 test: added pred-lag columns ({len(PRED_LAG_COLS)})")

print(f"  Stage 2 test prediction (direct, ensemble of {N_FOLDS} folds)...")
X_test_s2 = test[feature_cols_s2]
pred_s2_lgb_test = np.mean([from_train_pred(m.predict(X_test_s2)) for m in models_s2_lgb], axis=0)
pred_s2_xgb_test = np.mean([from_train_pred(m.predict(X_test_s2)) for m in models_s2_xgb], axis=0)
pred_s2_cat_test = np.mean([from_train_pred(m.predict(X_test_s2)) for m in models_s2_cat], axis=0)

pred_s2_test = sum(
    w_s2[m] * {'lgb': pred_s2_lgb_test, 'xgb': pred_s2_xgb_test, 'cat': pred_s2_cat_test}[m]
    for m in best_s2_models
) / ws_s2
print(f"  Stage 2 test pred: mean={pred_s2_test.mean():.2f}  std={pred_s2_test.std():.2f}")

pred_s2_test_blend = pred_s1_test if use_s1_instead_of_s2 else pred_s2_test

pred_blend_test = best_alpha * pred_s1_test + (1 - best_alpha) * pred_s2_test_blend

pred = np.maximum(pred_blend_test, CLIP_PRED_MIN)
pred_hi = float(np.percentile(y_raw, 100 * best_clip_q))
pred = np.minimum(pred, pred_hi)

sub = pd.DataFrame({'ID': test['ID'], TARGET: pred})
save_path = os.path.join(project_root, 'submission_logtarget_s2_predlag_autoencoder_ae_es_wd.csv')
sub.to_csv(save_path, index=False)
print(f"▶ saved -> {save_path}")
print(f"\n▶▶ DONE - FINAL OOF MAE: {best_final_mae:.6f}")



  Predict test + submit (pred-lag S2: direct predict on test)
  Stage 1 test prediction (no lag)...
  Stage 1 test pred: mean=19.26  std=15.06
  Stage 2 test: added pred-lag columns (9)
  Stage 2 test prediction (direct, ensemble of 10 folds)...
  Stage 2 test pred: mean=19.77  std=16.02
▶ saved -> C:\SM\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition\submission_logtarget_s2_predlag_autoencoder_ae_es_wd.csv

▶▶ DONE - FINAL OOF MAE: 8.451366
